In [18]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler

In [19]:
CSV_PATH = "raw_loan_dataset.csv"
OUT_PATH = "clean_loan_dataset.csv"


In [20]:

# Load
df = pd.read_csv(CSV_PATH)

In [21]:
df = pd.read_csv(CSV_PATH)

# INITIAL HEAD
df.head()


,Income,CreditScore,EmploymentYears,LoanAmount,HasCollateral,PreviousDefaults,Approved
0,108810,537.0,1.1,25800,Yes,No,No
1,96482,524.0,15.0,11200,Y,No,Yes
2,28478,NaN,5.4,12100,No,No,Yes
3,"$25,851",561.0,17.6,7000,Yes,No,Yes
4,38396,527.0,9.8,10700,No,No,Approved


In [22]:
df.head()
df.info()
("   INITIAL MISSING VALUES   ")
df.isnull().sum()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 103 entries, 0 to 102
Data columns (total 7 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Income            98 non-null     object 
 1   CreditScore       97 non-null     float64
 2   EmploymentYears   98 non-null     float64
 3   LoanAmount        98 non-null     object 
 4   HasCollateral     101 non-null    object 
 5   PreviousDefaults  101 non-null    object 
 6   Approved          103 non-null    object 
dtypes: float64(2), object(5)
memory usage: 5.8+ KB


,0
Income,5
CreditScore,6
EmploymentYears,5
LoanAmount,5
HasCollateral,2
PreviousDefaults,2
Approved,0


In [23]:
# 2) Clean currency formatting
df["Income"] = df["Income"].replace(r"[\$,]", "", regex=True).astype(float)
df["LoanAmount"] = df["LoanAmount"].replace(r"[\$,]", "", regex=True).astype(float)
df


,Income,CreditScore,EmploymentYears,LoanAmount,HasCollateral,PreviousDefaults,Approved
0,108810.0,537.0,1.1,25800.0,Yes,No,No
1,96482.0,524.0,15.0,11200.0,Y,No,Yes
2,28478.0,NaN,5.4,12100.0,No,No,Yes
3,25851.0,561.0,17.6,7000.0,Yes,No,Yes
4,38396.0,527.0,9.8,10700.0,No,No,Approved
...,...,...,...,...,...,...,...
98,97529.0,629.0,17.6,15900.0,No,No,No
99,62268.0,720.0,12.2,NaN,Yes,No,Yes
100,108810.0,537.0,1.1,25800.0,Yes,No,No
101,96482.0,524.0,15.0,11200.0,Y,No,Yes


In [24]:
#  BEFORE imputation
yes_no_map = {
    "yes": "Yes", "y": "Yes", "yse": "Yes", "1": "Yes", "approved": "Yes",
    "no": "No", "n": "No", "0": "No", "rejected": "No",
}

df["HasCollateral"] = df["HasCollateral"].astype(str).str.strip().str.lower().replace(yes_no_map).replace({"nan": np.nan})
df["PreviousDefaults"] = df["PreviousDefaults"].astype(str).str.strip().str.lower().replace(yes_no_map).replace({"nan": np.nan})
df["Approved"] = df["Approved"].astype(str).str.strip().str.lower().replace(yes_no_map).replace({"nan": np.nan})

In [25]:
#  Impute missing values
# --------------------------------
df["Income"] = df["Income"].fillna(df["Income"].median())
df["CreditScore"] = df["CreditScore"].fillna(df["CreditScore"].median())
df["EmploymentYears"] = df["EmploymentYears"].fillna(df["EmploymentYears"].median())
df["LoanAmount"] = df["LoanAmount"].fillna(df["LoanAmount"].median())
df["HasCollateral"] = df["HasCollateral"].fillna(df["HasCollateral"].mode()[0])
df["PreviousDefaults"] = df["PreviousDefaults"].fillna(df["PreviousDefaults"].mode()[0])

In [26]:
#   duplicates Remove
before = df.shape[0]
df = df.drop_duplicates()
print(f"\nDropped duplicates: {before} -> {df.shape[0]} rows")


Dropped duplicates: 103 -> 100 rows


In [27]:
# 6) IQR capping on numeric columns
def iqr_fun(series, k=1.5):
    q1, q3 = series.quantile([0.25, 0.75])
    iqr = q3 - q1
    lower = q1 - k * iqr
    upper = q3 + k * iqr
    return lower, upper

low_income,  high_income  = iqr_fun(df["Income"])
low_credit,  high_credit  = iqr_fun(df["CreditScore"])
low_loan,    high_loan    = iqr_fun(df["LoanAmount"])
low_emp,     high_emp     = iqr_fun(df["EmploymentYears"])

df.loc[:, "Income"]          = df["Income"].clip(lower=low_income,  upper=high_income)
df.loc[:, "CreditScore"]     = df["CreditScore"].clip(lower=low_credit, upper=high_credit)
df.loc[:, "LoanAmount"]      = df["LoanAmount"].clip(lower=low_loan,    upper=high_loan)
df.loc[:, "EmploymentYears"] = df["EmploymentYears"].clip(lower=low_emp,     upper=high_emp)

In [28]:
# Label encoding (Yes/No → 0/1)
# same mapping for target (Approved) and two-category features
# Approved=1, Rejected=0
# --------------------------------
df["Approved"] = df["Approved"].map({"Yes": 1, "No": 0}).astype(int)

print("\n=== CLASS DISTRIBUTION (after label encoding) ===")
print(df["Approved"].value_counts())
print(df["Approved"].value_counts(normalize=True).round(3))


=== CLASS DISTRIBUTION (after label encoding) ===
Approved
1    66
0    34
Name: count, dtype: int64
Approved
1    0.66
0    0.34
Name: proportion, dtype: float64


/tmp/ipykernel_3622/3063273752.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["Approved"] = df["Approved"].map({"Yes": 1, "No": 0}).astype(int)


In [29]:
# Class balance check

class_ratio = df["Approved"].value_counts(normalize=True).min()
if class_ratio < 0.30:
    print("\nWarning: severely imbalanced classes — consider balancing techniques (L3).")
else:
    print("\nClass balance OK for baseline Accuracy (both classes well represented).")

df["HasCollateral"] = df["HasCollateral"].map({"Yes": 1, "No": 0}).astype(int)
df["PreviousDefaults"] = df["PreviousDefaults"].map({"Yes": 1, "No": 0}).astype(int)


Class balance OK for baseline Accuracy (both classes well represented).


/tmp/ipykernel_3622/1801993507.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["HasCollateral"] = df["HasCollateral"].map({"Yes": 1, "No": 0}).astype(int)
/tmp/ipykernel_3622/1801993507.py:11: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["PreviousDefaults"] = df["PreviousDefaults"].map({"Yes": 1, "No": 0}).astype(int)


In [30]:
# 9) Feature engineering (no leakage from label)
df["DebtToIncome"] = df["LoanAmount"] / df["Income"].replace(0, np.nan)
df["IncomePerYearEmployed"] = df["Income"] / (df["EmploymentYears"] + 1)

/tmp/ipykernel_3622/528647547.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["DebtToIncome"] = df["LoanAmount"] / df["Income"].replace(0, np.nan)
/tmp/ipykernel_3622/528647547.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["IncomePerYearEmployed"] = df["Income"] / (df["EmploymentYears"] + 1)


In [31]:
# MinMaxScaler on numeric features
# scale after outlier capping so mean/std are not skewed by extremes
binary_cols = {"HasCollateral", "PreviousDefaults", "Approved"}
numeric_cols = df.select_dtypes(include=["int64", "float64"]).columns.tolist()
scale_cols = [c for c in numeric_cols if c not in binary_cols]

scaler = MinMaxScaler()
df.loc[:, scale_cols] = scaler.fit_transform(df[scale_cols])

In [32]:
# 11) Final snapshot + save
# Features (X) = all columns except Approved; label (y) = Approved
print("=== FINAL HEAD ===")
print(df.head())

print("FINAL INFO ")
print(df.info())

print("FINAL MISSING VALUES")
print(df.isnull().sum())

df.to_csv(OUT_PATH, index=False)
print(f"nSaved cleaned dataset to {OUT_PATH}")

=== FINAL HEAD ===
     Income  CreditScore  EmploymentYears  LoanAmount  HasCollateral  \
0  0.585174     0.121560         0.024490    0.339800              1   
1  0.498215     0.091743         0.591837    0.101287              1   
2  0.018530     0.353211         0.200000    0.115989              0   
3  0.000000     0.176606         0.697959    0.032673              1   
4  0.088490     0.098624         0.379592    0.093118              0   

   PreviousDefaults  Approved  DebtToIncome  IncomePerYearEmployed  
0                 0         0      0.258294               0.846035  
1                 0         1      0.104832               0.082051  
2                 0         1      0.496399               0.055679  
3                 0         1      0.300991               0.004620  
4                 0         1      0.310998               0.040752  
=== FINAL INFO ===
<class 'pandas.core.frame.DataFrame'>
Index: 100 entries, 0 to 99
Data columns (total 9 columns):
 #   Column      

In [33]:
df.head(40)

,Income,CreditScore,EmploymentYears,LoanAmount,HasCollateral,PreviousDefaults,Approved,DebtToIncome,IncomePerYearEmployed
0,0.585174,0.121560,0.024490,0.339800,1,0,0,0.258294,0.846035
1,0.498215,0.091743,0.591837,0.101287,1,0,1,0.104832,0.082051
2,0.018530,0.353211,0.200000,0.115989,0,0,1,0.496399,0.055679
3,0.000000,0.176606,0.697959,0.032673,1,0,1,0.300991,0.004620
4,0.088490,0.098624,0.379592,0.093118,0,0,1,0.310998,0.040752
5,0.307611,0.619266,0.126531,0.699204,1,0,1,0.830228,0.233398
6,0.565847,0.415138,0.575510,0.702471,0,0,1,0.531449,0.094887
7,0.067766,0.263761,0.865306,0.086584,0,0,1,0.325973,0.007493
8,0.336246,0.405963,0.491837,0.199306,1,0,1,0.254287,0.071967
9,0.220332,0.181193,0.461224,0.137227,1,0,1,0.255275,0.055849
